# Gate 0A cloud — Stage 0 + 1 + 2 (SoccerTrack v2)

```
DIAGNOSTIC — NOT OFFICIAL GATE 0A VERDICT
TEST SET UNTOUCHED
```

Thin orchestrator: clones the repository at the run branch, installs pinned deps,
and runs `python -m ml.gate0a.cloud.run_stage --stage all` (real detector + real
ReID + MOT + offline reconciliation on ONE VAL half, then the Oracle-vs-Real
bottleneck decomposition). All science lives in the repository; this notebook
only sequences it.

**Before running:** Kaggle Secrets `HF_TOKEN` (required; HF account with
SoccerTrack-v2 access) and optional `GH_PAT`. Settings: Accelerator **GPU**,
Internet **On**. Then **Save & Run All**. Re-runs resume from valid caches.

In [ ]:
# 1) Preflight (safe to print — no secrets)
import os
import shutil
import subprocess
import sys
import urllib.request

print("python", sys.version.split()[0])
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
try:
    import torch

    print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
except Exception as exc:
    print("torch import failed:", exc)
for p in ("/kaggle/working", "/kaggle/tmp"):
    if os.path.exists(p):
        u = shutil.disk_usage(p)
        print(f"disk {p}: free {u.free / 2**30:.1f} GiB / {u.total / 2**30:.1f} GiB")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
    print("internet: huggingface.co reachable")
except Exception as exc:
    print("internet check FAILED (enable Internet in notebook settings):", type(exc).__name__)

In [ ]:
# 2) Parameters
REPO_URL = "https://github.com/vansyson1308/aistockcounting"
BRANCH = "gate0a-cloud-stage12"
REPO_SHA = ""  # pin a commit for exact reproducibility; empty = branch head
REPO_DIR = "/kaggle/working/repo"
OUT = "/kaggle/working/gate0a_outputs"  # persisted outputs (20 GiB budget)
SCRATCH = "/kaggle/tmp/gate0a_scratch"  # dataset video + model caches (not persisted)

In [ ]:
# 3) Secrets → environment (values are never printed)
import os

try:
    from kaggle_secrets import UserSecretsClient

    _client = UserSecretsClient()
    for _name in ("HF_TOKEN", "GH_PAT"):
        try:
            _value = _client.get_secret(_name)
            if _value:
                os.environ[_name] = _value.strip()
        except Exception:
            pass
except Exception:
    pass
print("secrets present:", {n: bool(os.environ.get(n)) for n in ("HF_TOKEN", "GH_PAT")})

In [ ]:
# 4) Clone / checkout the exact code + install pinned dependencies
import os
import subprocess
import sys


def sh(cmd, cwd=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check).returncode


if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])
else:
    sh(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=False)
    sh(["git", "-C", REPO_DIR, "checkout", BRANCH], check=False)
    sh(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=False)
if REPO_SHA:
    sh(["git", "-C", REPO_DIR, "checkout", REPO_SHA])
sh(["git", "-C", REPO_DIR, "rev-parse", "HEAD"])
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/gate0a/cloud/requirements-cloud.txt"], cwd=REPO_DIR)

In [ ]:
# 5) Stage 0 → 1 → 2 (resumable; exit code 3 = OWNER UI ACTION REQUIRED)
import os
import subprocess
import sys

os.makedirs(OUT, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
rc = subprocess.run([sys.executable, "-m", "ml.gate0a.cloud.run_stage", "--stage", "all",
                     "--out", OUT, "--scratch", SCRATCH], cwd=REPO_DIR).returncode
print("run_stage exit code:", rc)
if rc == 3:
    print("CLOUD RUN PACKAGE: READY — waiting on owner UI authorization (HF access / HF_TOKEN).")
elif rc != 0:
    raise SystemExit(f"run_stage failed with exit code {rc} — see log above")

In [ ]:
# 6) Package reports (always) + optional GitHub push of the small report tree (GH_PAT)
import subprocess
import sys

subprocess.run([sys.executable, "-m", "ml.gate0a.cloud.publish", "--out", OUT, "--branch", BRANCH], cwd=REPO_DIR)

In [ ]:
# 7) Summary
from pathlib import Path

rep = Path(OUT) / "reports" / "gate0a" / "cloud" / "stage012"
for name in ("owner_action_required.md", "maturity.json", "executive_report.md"):
    p = rep / name
    if p.exists():
        print(f"\n===== {name} =====")
        print(p.read_text()[:8000])